# Infrastructure Setup

In [1]:
pip install --no-cache-dir -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 441.4/441.4 kB 40.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 42.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 37.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 279.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 243.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 84.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 100.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.8/171.8 kB 424.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 36.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 108.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 381.6 MB/s eta 0:00:00
     ━━━━━━━━━━

## Imports & Setup

In [2]:
import os
from typing import List, Dict, Any, Optional
from io import BytesIO
import re
import requests
import arxiv
import PyPDF2
from dotenv import load_dotenv
from langchain.agents.agent import AgentExecutor, AgentAction, AgentFinish
from langchain.agents import create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import Tool, StructuredTool
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
#from langgraph.prebuilt import ToolExecutor

#### Load env variables & assign it to variables

In [4]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# CHROMA_API_KEY = os.getenv("CHROMA_API_KEY")

In [6]:
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")
print("Groq API key loaded successfully!")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file")
print("OpenAI API key loaded successfully!")

Groq API key loaded successfully!
OpenAI API key loaded successfully!


# Create & define tools that our agent can use

In [8]:
# Simple ArXiv Search Tool
def arxiv_search(query, max_results=10):
    """Search for papers on arXiv and return basic info.
    """
    arxiv_client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=max_results)
    results = []
    for paper in arxiv_client.results(search):
        results.append({
            "id": paper.entry_id,
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            "url": paper.pdf_url,
            "abstract": paper.summary,
            "published": paper.published
        })
    return results

In [ ]:
# Download ArXiv paper as pdf and parse text
def download_paper_and_parse(paper_id):
    """Download a paper from arXiv by ID and extract text."""
    search = arxiv.Search(id_list=[paper_id])
    client = arxiv.Client()
    paper = next(client.results(search))
    response = requests.get(paper.pdf_url)
    pdf_file = BytesIO(response.content)
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()

    return text[:max_char]

In [ ]:
# Text Summarizer for the paper's contents
def paper_summarizer(text, groq_api_key=GROQ_API_KEY):
    """Summarize text using Groq."""
    model = ChatGroq(api_key=groq_api_key, model_name="meta-llama/llama-4-maverick-17b-128e-instruct")
    prompt = f"Summarize this academic paper: {text[:15000]}"
    return model.invoke(prompt).content